<div style="background-color: #FBF7EF; border: 1px solid #E5D9C3; border-radius: 14px; padding: 34px 30px; font-family: Georgia, serif; color: #3A322A;">
  <div style="letter-spacing: 3px; font-size: 12px; text-transform: uppercase; color: #B8923A; font-weight: bold;">Plataforma de Gestión &amp; Inteligencia Financiera para Spas</div>
  <h1 style="margin: 8px 0 6px 0; font-size: 38px; color: #2C251E; font-weight: normal;">.Finance</h1>
  <p style="margin: 0 0 18px 0; font-size: 16px; color: #6E5F50;">Planificador operativo de turnos, insumos, costos fijos mensuales (30 días), servicio más frecuente y proyección de ganancia.</p>
  <hr style="border: none; border-top: 1px solid #E5D9C3; margin: 18px 0;">
  <p style="margin: 0; font-size: 13px; color: #8C7B6B;"><b>Proyecto final · ADMI 1301 Introducción a la Programación · Universidad de los Andes · 2026-2</b><br>
  Integrantes: Matias Galeano (202523711) · Emily Rojas (202523724) · Paloma Restrepo (202410702) · Valeria Nuncio (202524226)</p>
</div>

## Cómo usar `.Finance`

Cualquier spa puede ingresar sus datos reales o explorar el caso base de **Sunny Side Luxury Spa**:

1. **Elige tu entrada en la Sección 1:**
   * **`"sunnyside"` (Entra a tu negocio):** carga automáticamente toda la operación, tarifas, costos fijos y equipo de Sunny Side Luxury Spa.
   * **`"nuevo_spa"` (Iniciar sesión / Nuevo spa):** te permite definir tu propia cantidad de servicios, precios, costos de insumos, lista de masajistas y tus **3 costos fijos mensuales** (`aseo`, `arriendo`, `servicios`).
2. **Ejecuta todas las celdas** (`Entorno de ejecución → Ejecutar todo`).
3. **Consulta tus paneles:** operación de la jornada, **servicio más frecuente**, asignación **automática o manual** de masajistas y comparación de escenarios.

In [ ]:
# Importamos las librerías básicas para tablas, gráficas y tarjetas visuales
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Paleta de colores del spa (códigos hexadecimales)
color_fondo = "#FBF7EF"      # crema marfil
color_tarjeta = "#FFFDF9"    # blanco cálido
color_borde = "#E5D9C3"      # arena suave
color_texto = "#2C251E"      # carbón cálido
color_oro = "#B8923A"        # dorado champán (para dar lujo)
color_taupe = "#8C7B6B"      # gris pardo
color_arena = "#C8B08A"      # beige dorado
color_salvia = "#5F8567"     # verde salvia (para la ganancia)
color_alerta = "#B35C44"     # terracota suave

# Función auxiliar: convierte un número en texto con formato de pesos colombianos.
# Ejemplo: formato_cop(2040000) devuelve "$2.040.000"
def formato_cop(valor):
    texto = f"{abs(valor):,.0f}"        # separa los miles con coma: 2,040,000
    texto = texto.replace(",", ".")     # cambia la coma por punto: 2.040.000
    if valor < 0:
        return "-$" + texto
    return "$" + texto

# Función visual: dibuja una tarjeta estética para mostrar los resultados del spa
def mostrar_tarjeta_lujo(titulo, subtitulo, filas_datos, destacado=""):
    filas_html = ""                     # acumula las filas de la tabla en HTML
    for etiqueta, valor in filas_datos:
        filas_html += (
            f"<tr>"
            f"<td style='padding: 9px 6px; border-bottom: 1px solid {color_borde}; color: {color_taupe}; font-size: 14px;'>{etiqueta}</td>"
            f"<td style='padding: 9px 6px; border-bottom: 1px solid {color_borde}; text-align: right; font-weight: bold; color: {color_texto}; font-size: 14px;'>{valor}</td>"
            f"</tr>"
        )

    bloque_destacado = ""
    if destacado != "":                 # si hay una nota final, la agrega abajo con borde dorado
        bloque_destacado = (
            f"<div style='margin-top: 14px; padding: 12px 14px; background-color: {color_fondo}; "
            f"border-left: 4px solid {color_oro}; border-radius: 6px; font-size: 14px; color: {color_texto};'>"
            f"{destacado}</div>"
        )

    # Armamos la tarjeta completa en HTML y la mostramos en pantalla
    html = (
        f"<div style='background-color: {color_tarjeta}; border: 1px solid {color_borde}; border-radius: 14px; "
        f"padding: 22px 26px; margin: 14px 0; font-family: Georgia, serif; max-width: 760px;'>"
        f"<div style='font-size: 11px; letter-spacing: 2px; text-transform: uppercase; color: {color_oro}; font-weight: bold;'>.Finance · Panel Ejecutivo</div>"
        f"<h3 style='margin: 4px 0 4px 0; color: {color_texto}; font-size: 22px; font-weight: normal;'>{titulo}</h3>"
        f"<p style='margin: 0 0 14px 0; color: {color_taupe}; font-size: 13px;'>{subtitulo}</p>"
        f"<table style='width: 100%; border-collapse: collapse;'>{filas_html}</table>"
        f"{bloque_destacado}"
        f"</div>"
    )
    display(HTML(html))

## 1 · Acceso a `.Finance` (Entra a tu negocio o registra un nuevo spa)

Selecciona el modo de entrada en la variable `modo_entrada`:
- **`"sunnyside"`**: abre directamente el perfil completo de **Sunny Side Luxury Spa**.
- **`"nuevo_spa"`**: permite que cualquier spa ingrese sus propios datos (cantidad variable de servicios, precios, costos de insumos, lista de masajistas y sus **3 costos fijos mensuales**).

In [ ]:
# ---------- SELECCIÓN DE ACCESO: "sunnyside" o "nuevo_spa" ----------
modo_entrada = "sunnyside"              # cambia a "nuevo_spa" para ingresar otro negocio
dias_mes = 30                           # base de 30 días para prorratear costos fijos mensuales

if modo_entrada == "sunnyside":
    # Datos precargados del caso base: Sunny Side Luxury Spa
    nombre_spa = "Sunny Side Luxury Spa"
    cabinas = 8                         # número de cabinas disponibles
    clientes_por_masajista = 3          # clientes que atiende cada masajista por turno
    clientes_por_recepcion = 10         # clientes que atiende cada persona de recepción
    costo_turno_masajista = 200000      # COP por turno de masajista
    costo_turno_recepcion = 120000      # COP por turno de recepción/apoyo

    # Costos fijos mensuales (30 días): solo estas 3 variables desglosadas
    aseo = 900000                       # COP al mes en aseo y lavandería
    arriendo = 6000000                  # COP al mes en arriendo del local
    servicios = 1200000                 # COP al mes en servicios públicos (agua, luz, gas)

    # Lista de masajistas disponibles en el spa
    masajistas = [
        "Valentina", "Camila", "Sofía", "Mariana",
        "Isabella", "Lucía", "Andrea", "Natalia"
    ]

    # Catálogo de servicios: listas paralelas (posición 0, 1, 2...)
    numero_servicios = 3                # cantidad total de servicios en el catálogo
    nombres_servicios = [
        "Masaje relajante",
        "Facial hidratante premium",
        "Ritual corporal"
    ]
    precios = [200000, 260000, 360000]  # COP por cliente para cada servicio
    costos_insumos = [30000, 50000, 90000]  # COP de insumos por cliente para cada servicio

elif modo_entrada == "nuevo_spa":
    # --- RELLENA AQUÍ LOS DATOS REALES DE TU SPA ---
    nombre_spa = "Mi Spa Boutique"
    cabinas = 5                         # cambia por tus cabinas reales
    clientes_por_masajista = 3          # clientes por turno de masajista
    clientes_por_recepcion = 10         # clientes por persona de recepción
    costo_turno_masajista = 160000      # pago por turno de masajista en COP
    costo_turno_recepcion = 110000      # pago por turno de recepción en COP

    # Costos fijos mensuales (30 días): rellena solo estas 3 variables
    aseo = 600000                       # gasto mensual de aseo
    arriendo = 3500000                  # gasto mensual de arriendo
    servicios = 850000                  # gasto mensual de servicios públicos

    # Lista de masajistas de tu spa
    masajistas = ["Laura", "Paula", "Daniela", "Sara", "Elena"]

    # Define cuántos servicios tiene tu spa y escríbelos en las listas paralelas
    numero_servicios = 4                # puedes poner 2, 3, 4, 5 o los que necesites
    nombres_servicios = [
        "Masaje descontracturante",
        "Limpieza facial profunda",
        "Drenaje linfático",
        "Circuito hidrotermal"
    ]
    precios = [150000, 180000, 170000, 220000]
    costos_insumos = [25000, 38000, 22000, 45000]

else:
    print("ERROR: Opción no válida. Escribe 'sunnyside' o 'nuevo_spa'.")

# ---------- CÁLCULOS AUTOMÁTICOS DE CAPACIDAD Y COSTOS FIJOS ----------
costosfijos = aseo + arriendo + servicios           # suma de los 3 costos fijos mensuales
costo_fijo_jornada = costosfijos / dias_mes         # costo fijo diario prorrateado en 30 días
capacidad_maxima = cabinas * clientes_por_masajista # capacidad máxima de clientes por jornada

# ---------- REVISIÓN DE COHERENCIA DE LAS LISTAS ----------
if len(nombres_servicios) != numero_servicios or len(precios) != numero_servicios or len(costos_insumos) != numero_servicios:
    print("ERROR: La cantidad de elementos en las listas debe coincidir con 'numero_servicios'.")
else:
    # Mostramos la tarjeta ejecutiva con el resumen del negocio cargado
    mostrar_tarjeta_lujo(
        titulo=nombre_spa,
        subtitulo=f"Sesión activa en .Finance · Base mensual de {dias_mes} días",
        filas_datos=[
            ("Cabinas habilitadas", f"{cabinas} cabinas"),
            ("Capacidad máxima por jornada", f"{capacidad_maxima} clientes"),
            ("Servicios activos en catálogo", f"{numero_servicios} tratamientos"),
            ("Masajistas registrados", f"{len(masajistas)} profesionales ({', '.join(masajistas)})"),
            ("Costos fijos mensuales (Aseo + Arriendo + Servicios)", formato_cop(costosfijos)),
            ("Costo fijo prorrateado por jornada (÷ 30 días)", formato_cop(costo_fijo_jornada)),
        ],
        destacado=f"Desglose mensual: Aseo {formato_cop(aseo)} · Arriendo {formato_cop(arriendo)} · Servicios públicos {formato_cop(servicios)}."
    )

# Construimos la tabla de servicios con su porcentaje de insumos y contribución
filas_catalogo = []
for i in range(numero_servicios):
    margen_unit = precios[i] - costos_insumos[i]                # ganancia bruta por cliente
    pct_insumo = (costos_insumos[i] / precios[i]) * 100         # porcentaje del precio que va a insumos
    filas_catalogo.append({
        "Servicio": nombres_servicios[i],
        "Precio": formato_cop(precios[i]),
        "Costo insumos": formato_cop(costos_insumos[i]),
        "Insumo (% precio)": f"{pct_insumo:.0f}%",
        "Contribución por cliente": formato_cop(margen_unit)
    })

display(pd.DataFrame(filas_catalogo))

## 2 · Motor de Cálculo y Predicciones Automáticas

El administrador **no necesita estimar porcentajes de demanda**: `.Finance` calcula automáticamente:
1. **Mezcla de demanda automática por precio:** los tratamientos más accesibles concentran mayor frecuencia de reserva y los más exclusivos menor frecuencia, manteniendo la suma exacta de clientes con redondeo `int(x + 0.5)`.
2. **Servicio más frecuente:** identifica automáticamente cuál tratamiento lidera la jornada.
3. **Asignación de masajistas (`"automatica"` o `"manual"`):** permite que el sistema asigne el equipo necesario desde la lista de masajistas o que el administrador elija manualmente quiénes cubren los turnos.

In [ ]:
# Función 1: cuántos trabajadores se necesitan (siempre redondeando hacia arriba)
def personal_necesario(clientes, capacidad):
    if clientes <= 0:
        return 0
    personal = clientes // capacidad          # división entera: cuántos grupos completos hay
    if clientes % capacidad > 0:              # % da el resto: si sobran clientes...
        personal = personal + 1               # ...hace falta un trabajador más
    return personal

# Función 2: calcula pesos automáticos de demanda según precio (más accesible = más frecuente)
def calcular_pesos_automaticos(lista_precios):
    inversos = []                             # guarda 1 / precio de cada servicio
    suma_inversos = 0
    for p in lista_precios:
        inv = 1 / p                           # a menor precio, mayor peso relativo
        inversos.append(inv)
        suma_inversos = suma_inversos + inv

    pesos = []                                # normaliza los pesos para que sumen 1 (100%)
    for inv in inversos:
        pesos.append(inv / suma_inversos)
    return pesos

# Función 3: reparte los clientes entre N servicios (cantidad variable)
def repartir_clientes_variable(clientes, lista_precios):
    n = len(lista_precios)                    # cantidad de servicios configurados
    pesos = calcular_pesos_automaticos(lista_precios)
    reparto = [0] * n                         # lista con ceros para guardar clientes por servicio
    asignados = 0

    # Asignamos del segundo servicio en adelante usando int(x + 0.5) para que todo .5 suba
    for i in range(1, n):
        cant = int(clientes * pesos[i] + 0.5)
        reparto[i] = cant
        asignados = asignados + cant

    # El primer servicio recibe el resto para conservar siempre el total exacto de clientes
    reparto[0] = clientes - asignados
    if reparto[0] < 0:
        reparto[0] = 0
    return reparto

# Función 4: encuentra el servicio más frecuente de la jornada
def obtener_servicio_mas_frecuente(nombres, reparto):
    mayor_clientes = -1                       # guarda la cantidad más alta encontrada
    indice_mayor = 0                          # guarda la posición del servicio ganador
    for i in range(len(reparto)):
        if reparto[i] > mayor_clientes:
            mayor_clientes = reparto[i]
            indice_mayor = i
    return nombres[indice_mayor], mayor_clientes

# Función 5: asigna masajistas en modo "automatica" o "manual"
def asignar_masajistas(lista_masajistas, requeridos, modo="automatica", seleccion_manual=None):
    if requeridos == 0:
        return [], "Sin turnos requeridos"

    if modo == "automatica":
        # Toma automáticamente los primeros masajistas disponibles de la lista
        asignados = lista_masajistas[:requeridos]
        if len(asignados) < requeridos:
            faltan = requeridos - len(asignados)
            estado = f"Asignación automática parcial: faltan {faltan} masajista(s) en la lista."
        else:
            estado = "Asignación automática completada según disponibilidad."
        return asignados, estado

    elif modo == "manual":
        # Verifica que los nombres elegidos manualmente existan en la lista y no se repitan
        if seleccion_manual is None:
            seleccion_manual = []
        asignados = []
        for nombre in seleccion_manual:
            if nombre in lista_masajistas and nombre not in asignados:
                asignados.append(nombre)
        if len(asignados) < requeridos:
            estado = f"Asignación manual incompleta: elegiste {len(asignados)} de {requeridos} requeridos."
        elif len(asignados) > requeridos:
            estado = f"Asignación manual con sobrepersonal: elegiste {len(asignados)} (se requieren {requeridos})."
        else:
            estado = "Asignación manual confirmada y exacta."
        return asignados, estado

    return [], "Modo de asignación no reconocido."

# Función 6: calcula todos los resultados de una jornada y los devuelve en un diccionario
def calcular_jornada_finance(clientes, modo_asignacion="automatica", seleccion_manual=None):
    # Si la demanda supera la capacidad máxima de cabinas, la operación no es viable
    if clientes > capacidad_maxima:
        return {
            "viable": False,
            "clientes": clientes,
            "sin_atender": clientes - capacidad_maxima,
        }

    # Calculamos personal requerido en masajistas y recepción
    masajistas_req = personal_necesario(clientes, clientes_por_masajista)
    recepcion_req = personal_necesario(clientes, clientes_por_recepcion)

    # Si es manual y el usuario eligió una lista, se paga según los elegidos
    if modo_asignacion == "manual" and seleccion_manual is not None and len(seleccion_manual) > 0:
        masajistas_pagados = len(seleccion_manual)
    else:
        masajistas_pagados = masajistas_req

    # Ejecutamos la asignación automática o manual de masajistas
    equipo_asignado, nota_equipo = asignar_masajistas(
        masajistas, masajistas_req, modo=modo_asignacion, seleccion_manual=seleccion_manual
    )

    # Repartimos los clientes entre los servicios y hallamos el servicio más frecuente
    reparto = repartir_clientes_variable(clientes, precios)
    servicio_top, clientes_top = obtener_servicio_mas_frecuente(nombres_servicios, reparto)

    # Sumamos ingresos y costos de insumos recorriendo todos los servicios
    ingresos = 0
    costo_insumos_total = 0
    for i in range(len(nombres_servicios)):
        ingresos = ingresos + reparto[i] * precios[i]
        costo_insumos_total = costo_insumos_total + reparto[i] * costos_insumos[i]

    # Costos totales incluyen personal, insumos y el costo fijo diario (mensual ÷ 30)
    costo_personal = masajistas_pagados * costo_turno_masajista + recepcion_req * costo_turno_recepcion
    costos_totales = costo_personal + costo_insumos_total + costo_fijo_jornada
    ganancia = ingresos - costos_totales

    # Margen neto y utilización de capacidad
    if ingresos > 0:
        margen = (ganancia / ingresos) * 100
    else:
        margen = 0

    if masajistas_pagados > 0:
        utilizacion = (clientes / (masajistas_pagados * clientes_por_masajista)) * 100
    else:
        utilizacion = 0

    return {
        "viable": True,
        "clientes": clientes,
        "masajistas_req": masajistas_req,
        "masajistas_pagados": masajistas_pagados,
        "recepcion_req": recepcion_req,
        "equipo_asignado": equipo_asignado,
        "nota_equipo": nota_equipo,
        "reparto": reparto,
        "servicio_top": servicio_top,
        "clientes_top": clientes_top,
        "ingresos": ingresos,
        "costo_personal": costo_personal,
        "costo_insumos": costo_insumos_total,
        "costo_fijo_jornada": costo_fijo_jornada,
        "costos_totales": costos_totales,
        "ganancia": ganancia,
        "proyeccion_mensual": ganancia * dias_mes,
        "margen": margen,
        "utilizacion": utilizacion,
    }

## 3 · Calculadora de la Jornada & Asignación de Masajistas

Define cuántos clientes esperas hoy (`clientes_esperados`) y decide si quieres asignar los masajistas de forma **`"automatica"`** o **`"manual"`**.

In [ ]:
# ---------- PARÁMETROS DE LA JORNADA (MODIFICABLES POR EL ADMINISTRADOR) ----------
clientes_esperados = 16                 # número de clientes esperados hoy

# Decide el modo de asignación de masajistas: "automatica" o "manual"
modo_asignacion = "automatica"          # cambia a "manual" si quieres elegirlos tú mismo

# Si eliges "manual", escribe aquí los nombres exactos de tu lista 'masajistas'
masajistas_manual = ["Valentina", "Camila", "Sofía", "Mariana", "Isabella", "Lucía"]

# Ejecutamos el cálculo de la jornada
res = calcular_jornada_finance(
    clientes=clientes_esperados,
    modo_asignacion=modo_asignacion,
    seleccion_manual=masajistas_manual
)

# Mostramos alerta si se excede la capacidad, o la tarjeta ejecutiva si es viable
if not res["viable"]:
    mostrar_tarjeta_lujo(
        titulo=f"Capacidad Excedida · {nombre_spa}",
        subtitulo="Alerta operativa de infraestructura",
        filas_datos=[
            ("Clientes solicitados", f"{res['clientes']} clientes"),
            ("Capacidad máxima del spa", f"{capacidad_maxima} clientes"),
            ("Clientes sin poder atender", f"{res['sin_atender']} clientes"),
        ],
        destacado="No es posible operar por encima de la capacidad máxima de cabinas."
    )
else:
    lista_equipo = ", ".join(res["equipo_asignado"]) if len(res["equipo_asignado"]) > 0 else "Ninguno"
    mostrar_tarjeta_lujo(
        titulo=f"Resumen Operativo · {clientes_esperados} Clientes",
        subtitulo=f"{nombre_spa} · Modo de asignación: {modo_asignacion.upper()}",
        filas_datos=[
            ("Servicio más frecuente", f"{res['servicio_top']} ({res['clientes_top']} clientes)"),
            ("Masajistas requeridos / asignados", f"{res['masajistas_req']} req. → {lista_equipo}"),
            ("Personal de recepción/apoyo", f"{res['recepcion_req']} persona(s)"),
            ("Utilización de cabina/masajista", f"{res['utilizacion']:.1f}%"),
            ("Ingresos estimados de la jornada", formato_cop(res["ingresos"])),
            ("Costo de personal", formato_cop(res["costo_personal"])),
            ("Costo de insumos", formato_cop(res["costo_insumos"])),
            ("Costo fijo diario (Aseo + Arriendo + Servicios ÷ 30)", formato_cop(res["costo_fijo_jornada"])),
            ("Costos totales de la jornada", formato_cop(res["costos_totales"])),
            ("Ganancia neta de la jornada", formato_cop(res["ganancia"])),
            ("Margen neto", f"{res['margen']:.1f}%"),
            ("Proyección mensual (30 jornadas iguales)", formato_cop(res["proyeccion_mensual"])),
        ],
        destacado=f"<b>Estado del equipo:</b> {res['nota_equipo']}"
    )

    # Tabla con el desglose de clientes, ingresos e insumos por cada servicio
    filas_desglose = []
    for i in range(len(nombres_servicios)):
        c_serv = res["reparto"][i]
        ing_serv = c_serv * precios[i]
        ins_serv = c_serv * costos_insumos[i]
        filas_desglose.append({
            "Servicio": nombres_servicios[i],
            "Clientes estimados": c_serv,
            "Ingresos": formato_cop(ing_serv),
            "Insumos": formato_cop(ins_serv),
            "Destacado": "★ Más frecuente" if nombres_servicios[i] == res["servicio_top"] else ""
        })
    display(pd.DataFrame(filas_desglose))

## 4 · Predicciones y Comparación de Escenarios

Proyección automática de **Baja afluencia (33%)**, **Ocupación media (67%)** y **Agenda llena (100%)** incluyendo el impacto de los costos fijos mensuales prorrateados a 30 días.

In [ ]:
# Definimos los tres escenarios de ocupación (33%, 67% y 100% de la capacidad máxima)
nombres_escenarios = ["Baja afluencia", "Ocupación media", "Agenda llena"]
clientes_escenarios = [
    max(1, capacidad_maxima // 3),          # un tercio de la capacidad
    max(1, (capacidad_maxima * 2) // 3),    # dos tercios de la capacidad
    capacidad_maxima                        # capacidad máxima completa
]

# Listas para construir la tabla comparativa y las barras de la gráfica
tabla_esc = []
ingresos_graf = []
costos_graf = []
ganancias_graf = []

# Calculamos cada escenario usando nuestra función principal
for idx in range(len(nombres_escenarios)):
    c_esc = clientes_escenarios[idx]
    r = calcular_jornada_finance(c_esc, modo_asignacion="automatica")
    ingresos_graf.append(r["ingresos"] / 1000000)         # pasamos a millones de COP
    costos_graf.append(r["costos_totales"] / 1000000)
    ganancias_graf.append(r["ganancia"] / 1000000)

    tabla_esc.append({
        "Escenario": nombres_escenarios[idx],
        "Clientes": c_esc,
        "Servicio más frecuente": r["servicio_top"],
        "Masajistas": r["masajistas_req"],
        "Ingresos": formato_cop(r["ingresos"]),
        "Costos Totales": formato_cop(r["costos_totales"]),
        "Ganancia Jornada": formato_cop(r["ganancia"]),
        "Proyección 30 días": formato_cop(r["proyeccion_mensual"]),
        "Margen": f"{r['margen']:.1f}%"
    })

display(pd.DataFrame(tabla_esc))

# ---------- GRÁFICA ESTÉTICA CON PALETA .FINANCE ----------
fig, ax = plt.subplots(figsize=(9, 4.8))
fig.patch.set_facecolor(color_fondo)        # fondo exterior crema marfil
ax.set_facecolor(color_tarjeta)             # fondo interior blanco cálido

x = [0, 1, 2]
ancho = 0.25                                # ancho de cada barra

# Dibujamos las 3 barras por escenario: ingresos, costos totales y ganancia neta
ax.bar([p - ancho for p in x], ingresos_graf, width=ancho, label="Ingresos", color=color_oro)
ax.bar(x, costos_graf, width=ancho, label="Costos Totales (incl. Fijos)", color=color_taupe)
ax.bar([p + ancho for p in x], ganancias_graf, width=ancho, label="Ganancia Neta", color=color_salvia)

# Detalles visuales limpios de la gráfica
ax.axhline(0, color=color_texto, linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(nombres_escenarios, fontsize=11, color=color_texto)
ax.set_ylabel("Millones de COP", fontsize=11, color=color_texto)
ax.set_title(f".Finance · Proyección por Escenario — {nombre_spa}", fontsize=13, color=color_texto, pad=14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(color_borde)
ax.spines["bottom"].set_color(color_borde)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()